# LLM + Rule-Based Hybrid Detector

A hybrid detection pipeline that combines generative span extraction with strict heuristic-based filtering. The framework uses an open-weights LLM (`Qwen/Qwen2.5-7B-Instruct`) in a zero-shot pipeline to extract exact substring candidates of hallucinations from the model's output.

Simultaneously, a regular-expression parser processes the structured context to extract verified "grounded" values from the tool output. The generated LLM spans are then dynamically evaluated: if an LLM-suggested hallucination span overlaps significantly (more than 30%) with the verified grounded character set, it is discarded. This hybrid approach aims to mitigate false positives and tighten span precision.

In [1]:
!pip install --upgrade pip

!pip install torch --index-url https://download.pytorch.org/whl/cu121

!pip install transformers accelerate pandas numpy tqdm ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.2 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 53.2 MB/s  0:00:13:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 62.5 MB/s  0:00:00m0:00:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 83.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 80.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 72.4 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 79.7 MB/s  0:00:05:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 94.4 MB/s  0:00:01:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 90.6 MB/s  0:00:00m0:00:0100:01
     ━━━

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["HF_HOME"] = "/app/.cache/huggingface"

import re
import json
import torch
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from transformers import pipeline

os.makedirs("/app/results", exist_ok=True)
DATA_DIR = Path("/app/data")

TEST_FILES = {
    "Combined": DATA_DIR / "combined_test.jsonl",
    "Type 1 + clean": DATA_DIR / "type1_test_balanced.jsonl",
    "Type 2 + clean": DATA_DIR / "type2_test_balanced.jsonl",
    "Type 3 + clean": DATA_DIR / "type3_test_balanced.jsonl",
}

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f: return [json.loads(line) for line in f if line.strip()]

def extract_grounded_spans(tool_output, answer):
    grounded_spans = []
    match = re.search(r'\{.*\}', tool_output.replace('\n', ' '))
    if match:
        try:
            data = json.loads(match.group(0))
            for k, v in data.items():
                if isinstance(v, (str, int, float)) and len(str(v)) >= 2:
                    for m in re.finditer(re.escape(str(v)), answer, re.IGNORECASE):
                        grounded_spans.extend(list(range(m.start(), m.end())))
        except: pass
    return set(grounded_spans)

In [3]:
print("load LLM (Qwen2.5-7B-Instruct)...")
llm = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct", model_kwargs={"torch_dtype": torch.bfloat16}, device_map="cuda:0")

def prompt_llm(sample):
    prompt = f"""You are a hallucination detector. 
Context (Tool output): {sample.get("context", "")}
User Query: {sample.get("query", "")}
Model Answer: {sample.get("output", "")}
Extract any exact sub-strings from the Model Answer that are hallucinated. Return ONLY a valid JSON array of strings, e.g. ["string1"]. If none, return []."""

    messages = [{"role": "user", "content": prompt}]
    res = llm(messages, max_new_tokens=128, truncation=True, return_full_text=False)[0]["generated_text"]
    
    spans = []
    arr_str = re.search(r'\[.*\]', res.replace('\n', ''))
    if arr_str:
        try:
            for phrase in json.loads(arr_str.group(0)):
                for m in re.finditer(re.escape(phrase), sample["output"]): spans.append({"start": m.start(), "end": m.end()})
        except: pass
    return spans

def hybrid_predict(sample):
    llm_spans = prompt_llm(sample)
    grounded = extract_grounded_spans(sample.get("context", ""), sample.get("output", ""))
    
    final_spans = []
    for span in llm_spans:
        span_chars = set(range(span["start"], span["end"]))
        if len(span_chars & grounded) > (len(span_chars) * 0.3): continue 
        final_spans.append(span)
    return final_spans

load LLM (Qwen2.5-7B-Instruct)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
def _char_set(spans): return {i for s in spans for i in range(int(s["start"]), int(s["end"]))}

def span_metrics(samples, pred_spans):
    micro_overlap = micro_pred = micro_gold = 0
    macro_f1 = []
    for sample, preds in zip(samples, pred_spans):
        gold = _char_set(sample.get("hallucination_labels", [])); pred = _char_set(preds)
        overlap = len(gold & pred)
        micro_overlap += overlap; micro_pred += len(pred); micro_gold += len(gold)
        if not pred and not gold: macro_f1.append(1.0); continue
        p = overlap / len(pred) if pred else 0.0; r = overlap / len(gold) if gold else 0.0
        macro_f1.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    p_mi = micro_overlap / micro_pred if micro_pred else 0.0
    r_mi = micro_overlap / micro_gold if micro_gold else 0.0
    f_mi = 2 * p_mi * r_mi / (p_mi + r_mi) if (p_mi + r_mi) > 0 else 0.0
    return {"P": p_mi, "R": r_mi, "F1": f_mi, "macro_F1": sum(macro_f1)/len(macro_f1) if macro_f1 else 0.0}

def example_metrics(samples, pred_spans):
    tp = fp = fn = 0
    for sample, preds in zip(samples, pred_spans):
        gold = 1 if sample.get("hallucination_labels") else 0; pred = 1 if preds else 0
        if gold and pred: tp += 1
        elif gold: fn += 1
        elif pred: fp += 1
    p = tp / (tp + fp) if (tp + fp) else 0.0; r = tp / (tp + fn) if (tp + fn) else 0.0
    return {"P": p, "R": r, "F1": 2 * p * r / (p + r) if (p + r) > 0 else 0.0}

results = []
for name, path in TEST_FILES.items():
    if not path.exists(): continue
    samples = read_jsonl(path)
    
    preds = [hybrid_predict(s) for s in tqdm(samples, desc=name)]
    sm, em = span_metrics(samples, preds), example_metrics(samples, preds)
    results.append({
        "Split": name, "N": len(samples),
        "Span P": round(sm["P"], 3), "Span R": round(sm["R"], 3), "Span F1": round(sm["F1"], 3), "Span macro F1": round(sm["macro_F1"], 3),
        "Ex P": round(em["P"], 3), "Ex R": round(em["R"], 3), "Ex F1": round(em["F1"], 3)
    })

with open("/app/results/llm_rulebased_metrics.json", "w", encoding="utf-8") as f: json.dump(results, f, indent=4)
display(pd.DataFrame(results))

Combined:   0%|          | 0/599 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=128) and `max_len

Type 1 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Type 2 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Type 3 + clean:   0%|          | 0/299 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,Split,N,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
0,Combined,599,0.463,0.120,0.191,0.337,0.848,0.535,0.656
1,Type 1 + clean,300,0.236,0.453,0.310,0.556,0.727,0.800,0.762
2,Type 2 + clean,300,0.408,0.100,0.161,0.408,0.611,0.460,0.525
3,Type 3 + clean,299,0.296,0.070,0.113,0.380,0.535,0.362,0.432
